In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import pickle

SEED=42
np.random.seed(SEED)

In [21]:
cohort_name = "mimic_cohort_NF_30_days"

feat_ts_path = f"/home/christel-sirocchi/GitHub/FLABNET/flabnet-pipeline/MIMIC_IV/saved_data/processed_admission_features_for_ts/{cohort_name}/{cohort_name}_admissions_labs_14_days_to_ts.csv.gz"
feat = pd.read_csv(feat_ts_path)

cohort_path = f"/home/christel-sirocchi/GitHub/FLABNET/flabnet-pipeline/MIMIC_IV/saved_data/cohorts/{cohort_name}.csv.gz"
cohort = pd.read_csv(cohort_path)

fold_path = f"/home/christel-sirocchi/GitHub/FLABNET/flabnet-pipeline/MIMIC_IV/saved_data/folds/{cohort_name}/fold_0.pkl"
train_ids_, val_ids_, test_ids_ = pickle.load(open(fold_path, "rb"))

path_pickle = "/home/christel-sirocchi/GitHub/DATASETS/Physionet_2012/preprocessed_strats/physionet_2012.pkl"
ts, oc, train_ids, val_ids, test_ids = pickle.load(open(path_pickle, "rb"))

def get_counts(df, ids):
    sub = df[df["hadm_id"].isin(ids)]
    n_pos = (sub["label"] == 1).sum()
    n_neg = (sub["label"] == 0).sum()
    return n_pos, n_neg

train_pos_target, train_neg_target = get_counts(cohort, train_ids_[:,1])
val_pos_target, val_neg_target     = get_counts(cohort, val_ids_[:,1])

print("\n🎯 Target counts:")
print(f"TRAIN → pos: {train_pos_target}, neg: {train_neg_target}")
print(f"VAL   → pos: {val_pos_target}, neg: {val_neg_target}")

oc_train = oc[oc.ts_id.isin(train_ids)]
pos_train_ids = oc_train[oc_train["in_hospital_mortality"] == 1]["ts_id"].values
neg_train_ids = oc_train[oc_train["in_hospital_mortality"] == 0]["ts_id"].values
train_pos = np.random.choice(pos_train_ids, train_pos_target, replace=False)
train_neg = np.random.choice(neg_train_ids, train_neg_target, replace=False)
train_sub = np.concatenate([train_pos, train_neg])

oc_val = oc[oc.ts_id.isin(val_ids)]
pos_val_ids = oc_val[oc_val["in_hospital_mortality"] == 1]["ts_id"].values
neg_val_ids = oc_val[oc_val["in_hospital_mortality"] == 0]["ts_id"].values
val_pos = np.random.choice(pos_val_ids, val_pos_target, replace=False)
val_neg = np.random.choice(neg_val_ids, val_neg_target, replace=False)
val_sub = np.concatenate([val_pos, val_neg])

test_sub = test_ids.copy()

keep_ids = np.concatenate([train_sub, val_sub, test_sub])
ts_sub = ts[ts.ts_id.isin(keep_ids)].reset_index(drop=True)
oc_sub = oc[oc.ts_id.isin(keep_ids)].reset_index(drop=True)

def print_stats(name, ids):
    subset = oc_sub[oc_sub.ts_id.isin(ids)]
    ratio = subset["in_hospital_mortality"].mean()
    print(f"{name}: N={len(ids)}, pos_ratio={ratio:.3f}")

print("\n📊 Final splits:")
print_stats("TRAIN", train_sub)
print_stats("VAL", val_sub)
print_stats("TEST (unchanged)", test_sub)


ts = ts_sub
oc = oc_sub
train_ids = train_sub
val_ids = val_sub
test_ids = test_sub


demo_features = ["Age", "Gender", "Height", "ICUType_1", "ICUType_2", "ICUType_3", "ICUType_4"]
demo = ts[ts["variable"].isin(demo_features)]
ts = ts[~ts["variable"].isin(demo_features)]

feat = pd.read_csv(feat_ts_path)
freq = feat.groupby(["hadm_id","itemid"]).size().mean()
freqphy = ts.groupby(["ts_id","variable"]).size().mean()
pct = (freq/freqphy)*100

ts_sub_parts = []

print(f"Mean measurements per (ts_id, var) in MIMIC: {freq:.2f}")

for (ts_id, var), group in ts.groupby(["ts_id", "variable"]):
    rows_before = len(group)
    rows_kept = max(1, int(np.ceil(rows_before * pct / 100)))
    rows_removed = rows_before - rows_kept

    sampled = group.sample(n=rows_kept, random_state=SEED)
    ts_sub_parts.append(sampled)

ts_sub = pd.concat(ts_sub_parts, ignore_index=True)

print(f"Mean measurements per (ts_id, var) after sparsification: {ts_sub.groupby(['ts_id','variable']).size().mean():.2f}")

ts = (pd.concat([demo, ts_sub]).sort_values(by=["ts_id", "minute"]).reset_index(drop=True))


🎯 Target counts:
TRAIN → pos: 160, neg: 3229
VAL   → pos: 43, neg: 835

📊 Final splits:
TRAIN: N=3389, pos_ratio=0.047
VAL: N=878, pos_ratio=0.049
TEST (unchanged): N=3997, pos_ratio=0.139
Mean measurements per (ts_id, var) after sparsification: 6.24
Mean measurements per (ts_id, var) after sparsification: 6.77


In [22]:
with open("physionet_2012_multiperturbed_OM.pkl", "wb") as f:
    pickle.dump(
        [ts, oc, train_ids, val_ids, test_ids],
        f
    )

In [23]:
cohort = "mimic_cohort_NF_30_days"

feat_ts_path = f"/home/christel-sirocchi/GitHub/FLABNET/flabnet-pipeline/MIMIC_IV/saved_data/processed_admission_features_for_ts/{cohort}/{cohort}_admissions_labs_14_days_to_ts.csv.gz"
cohort_path = f"/home/christel-sirocchi/GitHub/FLABNET/flabnet-pipeline/MIMIC_IV/saved_data/cohorts/{cohort}.csv.gz"
fold_path = f"/home/christel-sirocchi/GitHub/FLABNET/flabnet-pipeline/MIMIC_IV/saved_data/folds/{cohort}/fold_0.pkl"

path_pickle = "/home/christel-sirocchi/GitHub/DATASETS/Physionet_2012/preprocessed_strats/physionet_2012.pkl"

filepath = os.path.join(path_pickle)
ts, oc, train_ids, val_ids, test_ids = pickle.load(open(filepath, "rb"))
SEED=42
np.random.seed(SEED)


# PERTURBATION 1 - SUBSAMPLING

train_ids_, val_ids_, test_ids_ = pickle.load(open(fold_path, "rb"))

N_train = train_ids_.shape[0]
N_val = val_ids_.shape[0]
N_test = test_ids_.shape[0]

train_sub = np.random.choice(train_ids, N_train, replace=False)
val_sub   = np.random.choice(val_ids, N_val, replace=False)
keep_ids  = np.concatenate([train_sub, val_sub])

ts_sub = ts[ts.ts_id.isin(np.concatenate([keep_ids, test_ids]))].reset_index(drop=True)
oc_sub = oc[oc.ts_id.isin(np.concatenate([keep_ids, test_ids]))].reset_index(drop=True)

print("\n📊 Subset composition:")
print(f"   Original train: {len(train_ids)} → Subset train: {len(train_sub)}")
print(f"   Original valid: {len(val_ids)} → Subset valid: {len(val_sub)}")
ts = ts_sub
oc = oc_sub
train_ids = train_sub
val_ids = val_sub

# PERTURBATION 2 - UNBALANCED

cohort = pd.read_csv(cohort_path)
pct = cohort.label.mean()*100

lengths = oc[oc.ts_id.isin(train_ids)]["length_of_stay"].values
T = np.percentile(lengths, 100 - pct)
oc["unbalanced"] = (oc["length_of_stay"] > T).astype(int)

print(f"Threshold length of stay: {T:.2f} days")

for split_name, split_ids in zip(["train", "val", "test"], [train_ids, val_ids, test_ids]):
    split_oc = oc[oc.ts_id.isin(split_ids)]
    counts = split_oc["unbalanced"].value_counts(normalize=True) * 100
    print(f"{split_name.upper()} split:")
    print(f"  Class 0: {counts.get(0,0):.2f}%, Class 1: {counts.get(1,0):.2f}%")


# PERTURBATION 3 - SUBSAMPLED

demo_features = ["Age", "Gender", "Height", "ICUType_1", "ICUType_2", "ICUType_3", "ICUType_4"]
demo = ts[ts["variable"].isin(demo_features)]
ts = ts[~ts["variable"].isin(demo_features)]

feat = pd.read_csv(feat_ts_path)
freq = feat.groupby(["hadm_id","itemid"]).size().mean()
freqphy = ts.groupby(["ts_id","variable"]).size().mean()
pct = (freq/freqphy)*100

ts_sub_parts = []

print(freq)

for (ts_id, var), group in ts.groupby(["ts_id", "variable"]):
    rows_before = len(group)
    rows_kept = max(1, int(np.ceil(rows_before * pct / 100)))
    rows_removed = rows_before - rows_kept

    sampled = group.sample(n=rows_kept, random_state=SEED)
    ts_sub_parts.append(sampled)

ts_sub = pd.concat(ts_sub_parts, ignore_index=True)

print(ts_sub.groupby(["ts_id","variable"]).size().mean())

ts = (pd.concat([demo, ts_sub]).sort_values(by=["ts_id", "minute"]).reset_index(drop=True))

### SAVE

with open("physionet_2012_multiperturbed_LOS.pkl", "wb") as f:
    pickle.dump(
        [ts, oc, train_ids, val_ids, test_ids],
        f
    )



📊 Subset composition:
   Original train: 6392 → Subset train: 3389
   Original valid: 1599 → Subset valid: 878
Threshold length of stay: 35.00 days
TRAIN split:
  Class 0: 95.31%, Class 1: 4.69%
VAL split:
  Class 0: 95.79%, Class 1: 4.21%
TEST split:
  Class 0: 94.92%, Class 1: 5.08%
6.242977278148629
6.748762409839181
